In [1]:
import pandas as pd
import re

In [4]:
# Load the Excel file
df = pd.read_excel("data/uni_infodisclosure_reports.xlsx")

# remove header and appendix

def clean_report_text(text):
    if not isinstance(text, str):
        return ""

    # ---------- 1. row ----------
    text = text.replace("\r", "\n")

    # ---------- 2. remove header ----------
    header_patterns = [
        r"当前位置[:：].*?\n",
        r"来源[:：].*?\n",
        r"发布时间[:：].*?\n",
        r"作者[:：].*?\n",
        r"校对[:：].*?\n",
        r"审核[:：].*?\n"
    ]
    for p in header_patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    # ---------- 3. remove after “附件” ----------
    appendix_match = re.search(r"\n\s*附件", text)
    if appendix_match:
        text = text[:appendix_match.start()]

    # ---------- 4. 删除 footer ----------
    footer_patterns = [
        r"上一条.*",
        r"下一条.*",
        r"版权所有.*",
        r"Copyright.*",
        r"电话[:：].*",
        r"【关闭】.*"
    ]
    for p in footer_patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    # ---------- 5. remove URL and weblinks ----------
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"\S+\.htm[l]?", "", text)

    # ---------- 6. 行级过滤：只保留“像正文的行” ----------
    lines = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue

        # 中文比例过滤（至少30%是中文）
        chinese_chars = re.findall(r"[\u4e00-\u9fff]", line)
        if len(chinese_chars) / max(len(line), 1) < 0.3:
            continue

        # URL 密度过滤
        if line.count("/") > 3:
            continue

        lines.append(line)

    return "\n".join(lines)


def split_by_char_count(text, chars_per_section=200, overlap=0):
    """
    Split text into sections by character count.

    Args:
        text: Input text string
        chars_per_section: Target number of characters per section (default 100)
        overlap: Number of characters to overlap between sections (default 0)

    Returns:
        List of text sections
    """
    if not isinstance(text, str) or not text.strip():
        return []

    text = text.strip()
    sections = []

    start = 0
    while start < len(text):
        # Extract section
        end = start + chars_per_section
        section = text[start:end].strip()

        if section:
            sections.append(section)

        # Move to next position (with overlap if specified)
        start = end - overlap

    return sections



# Create new rows
new_rows = []

for idx, row in df.iterrows():
    text = row['text']
    doc_id = row['doc_id']

    # Split by character count (adjust chars_per_section as needed)
    sections = split_by_char_count(text, chars_per_section=400, overlap=0)

    if len(sections) == 0:
        print(f"Skipping doc_id {doc_id}: empty text")
        continue

    # Create a new row for each section
    for section_num, section_text in enumerate(sections, start=1):
        new_row = row.copy()
        new_row['doc_id'] = f"{doc_id}.{section_num}"
        new_row['text'] = section_text
        new_rows.append(new_row)

# Create new DataFrame
df_cleaned_split = pd.DataFrame(new_rows)
df_cleaned_split.to_csv("cleaned_split_snippets_utf8sig.csv", index=False, encoding="utf-8-sig")


Skipping doc_id 1: empty text
Skipping doc_id 10: empty text
Skipping doc_id 13: empty text
Skipping doc_id 16: empty text
Skipping doc_id 17: empty text
Skipping doc_id 18: empty text
Skipping doc_id 19: empty text
Skipping doc_id 22: empty text
Skipping doc_id 23: empty text
Skipping doc_id 24: empty text
Skipping doc_id 25: empty text
Skipping doc_id 26: empty text
Skipping doc_id 27: empty text
Skipping doc_id 28: empty text
Skipping doc_id 31: empty text
Skipping doc_id 32: empty text
Skipping doc_id 33: empty text
Skipping doc_id 34: empty text
Skipping doc_id 35: empty text
Skipping doc_id 37: empty text
Skipping doc_id 38: empty text
Skipping doc_id 39: empty text
Skipping doc_id 40: empty text
Skipping doc_id 41: empty text
Skipping doc_id 42: empty text
Skipping doc_id 46: empty text
Skipping doc_id 47: empty text
Skipping doc_id 48: empty text
Skipping doc_id 53: empty text
Skipping doc_id 54: empty text
Skipping doc_id 55: empty text
Skipping doc_id 56: empty text
Skipping 